# Recon 7
Metrics dump, cluster service scan, neighbor catch.

In [ ]:
import subprocess
def run(cmd, t=40):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("curl -s --max-time 10 http://127.0.0.1:9090/metrics | head -120", 15))
print(run("curl -s --max-time 10 http://127.0.0.1:8012/healthz | head -5", 15))

In [ ]:
import subprocess
script = r'''
import socket, concurrent.futures
def probe(ip, ports):
    out = []
    for port in ports:
        try:
            s = socket.create_connection((ip, port), timeout=0.3)
            out.append(port); s.close()
        except Exception:
            pass
    return (ip, out) if out else None
ports = [80, 443, 8012, 8112, 9090, 3838, 5000, 8080]
ips = []
for a in [0,1]:
    for b in range(1,255):
        ips.append("10.96.%d.%d" % (a, b))
with concurrent.futures.ThreadPoolExecutor(150) as ex:
    for r in ex.map(lambda ip: probe(ip, ports), ips):
        if r: print("OPEN", r)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=170)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])

In [ ]:
import subprocess
script = r'''
import socket, urllib.request, concurrent.futures
def chk(ip):
    res = []
    for port in [8012, 9090, 3838]:
        try:
            s = socket.create_connection((ip, port), timeout=0.5)
            res.append(port); s.close()
        except Exception:
            pass
    return (ip, res) if res else None
found = []
with concurrent.futures.ThreadPoolExecutor(80) as ex:
    for r in ex.map(chk, ["192.168.4.%d" % i for i in range(1,255) if i != 124]):
        if r:
            print("FOUND", r)
            found.append(r)
# immediately probe found neighbors
def fetch(ip, port, path):
    try:
        resp = urllib.request.urlopen(urllib.request.Request("http://%s:%d%s" % (ip, port, path), headers={"User-Agent":"Mozilla/5.0"}), timeout=4)
        print(ip, port, path, "->", resp.status, resp.read(300)[:200])
    except Exception as e:
        print(ip, port, path, "FAIL:", e)
for ip, ports in found:
    if 8012 in ports:
        fetch(ip, 8012, "/")
        fetch(ip, 8012, "/healthz")
    if 9090 in ports:
        fetch(ip, 9090, "/metrics")
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=170)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])

In [ ]:
import subprocess
script = r'''
import socket
# does 8012 proxy based on Host header to the share domain?
def req(port, path, host):
    try:
        s = socket.create_connection(("127.0.0.1", port), timeout=3)
        s.send(("GET %s HTTP/1.0\r\nHost: %s\r\n\r\n" % (path, host)).encode())
        d = s.recv(600)
        s.close()
        print(port, path, host, "->", d[:250])
    except Exception as e:
        print(port, path, host, "FAIL:", e)
hosts = ["01a00558-fe84-93de-d102-439343d3ec1f.share.connect.posit.cloud", "127.0.0.1", "example.com"]
for h in hosts:
    req(8012, "/", h)
    req(8012, "/healthz", h)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=90)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])